# 🍀 klover — Train the four-leaf clover detector (Colab)

This notebook goes **public dataset → YOLOv8 training → `clover.onnx` export** in one run.

**Prerequisites (all free):**
1. Open this notebook in [Google Colab](https://colab.research.google.com)
2. `Runtime > Change runtime type > T4 GPU` (free GPU)
3. Sign up at [Roboflow](https://roboflow.com) (free) → copy your **Private API Key** from `Settings > API Keys`

Then run each cell top to bottom. When it finishes, `clover.onnx` downloads automatically.

## 1. Install dependencies

In [ ]:
!pip -q install ultralytics roboflow onnx onnxruntime
import torch
print('CUDA (GPU) available:', torch.cuda.is_available())  # should be True

## 2. Download the dataset (Roboflow)

Paste your API key into `API_KEY` below. The default is the public dataset **"4 Leaf Clover Detect"** (1,985 images, 4-leaf / 5-leaf).

> To use a different dataset, paste the snippet from that Roboflow Universe page's **`Download Dataset > YOLOv8 > show download code`** (just change workspace/project/version).

In [ ]:
from roboflow import Roboflow

API_KEY = "PASTE_YOUR_ROBOFLOW_API_KEY_HERE"  # ← paste your key

rf = Roboflow(api_key=API_KEY)
# Default example dataset (swap workspace/project/version if needed)
project = rf.workspace("test-ara07").project("4-leaf-clover-detect")
dataset = project.version(1).download("yolov8")
print("dataset location:", dataset.location)

In [ ]:
# Print class names and indices -> later set the '4-leaf' index in the app's TARGET_CLASS_INDEX.
import yaml, os
with open(os.path.join(dataset.location, 'data.yaml')) as f:
    data_yaml = yaml.safe_load(f)
print('classes:', data_yaml['names'])
for i, name in enumerate(data_yaml['names']):
    print(f'  index {i} = {name}')

## 3. Train (YOLOv8 nano)

`imgsz=640` must match the app (`YoloConfig.INPUT_SIZE`). More `epochs` = higher accuracy but longer training (50 is a fine start).

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # smallest, fastest model (good for mobile)
model.train(
    data=os.path.join(dataset.location, 'data.yaml'),
    imgsz=640,
    epochs=50,
    batch=16,
    name='klover',
)

## 4. Export to ONNX

NMS runs in the app, so export with `nms=False` (default). The output shape is `[1, 4+numClasses, 8400]`, matching the app's `decodeYoloOutput`.

In [ ]:
best = model.trainer.best  # path to the best weights
print('best weights:', best)
onnx_path = YOLO(best).export(format='onnx', imgsz=640, opset=12)
print('exported ONNX:', onnx_path)

In [ ]:
# Rename to clover.onnx and download
import shutil
from google.colab import files
shutil.copy(onnx_path, 'clover.onnx')
files.download('clover.onnx')

## 5. Add it to the app

1. Put the downloaded **`clover.onnx`** into
   `shared/src/commonMain/composeResources/files/clover.onnx`.
2. If the dataset has multiple classes (e.g. 3/4/5-leaf), set the **4-leaf index** you saw in step 2
   into `shared/.../detection/YoloConfig.kt` `TARGET_CLASS_INDEX`.
   (For a single 4-leaf-only class, leave it at `0`.)
3. Rebuild/run the Android app — **real on-device detection** now works. 🎉